# v3: the 3.3B translator rerun

The v2 run failed on `MT 3b` with CUDA OOM, which killed all ten `pivot_3b_*` evaluations.
Every `pivot_600m_*` run succeeded and is already in `results_v2.zip`, so this version runs
**only the 3B strategies** (5 runs instead of 16).

What changed:
- `dtype=` replaces the deprecated `torch_dtype=` when loading NLLB (the old name is ignored
  by current transformers), plus an `assert` that fails loudly if fp16 does not take effect
- 3B batch size 16 → 8, with `del enc, g` and a periodic `empty_cache()`
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` set before torch imports
- `SMOKE = False`, output goes to `results_v3.zip`

**Settings: GPU T4 x2, Internet On.** Expect roughly 1–1.5 h.

**Watch the line `loaded facebook/nllb-200-3.3B: N GiB`.** About 6.6 means fp16 is working and
the original failure was activation memory; about 13 means `torch_dtype` was being ignored and
the weights were fp32. Either way it should now fit — but tell me which number you see, because
it decides whether the paper can still claim the translator ran in fp16.

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate sentencepiece bitsandbytes

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # set BEFORE torch loads
import re, gc, time, json, zipfile, shutil, traceback
import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, BitsAndBytesConfig

SMOKE = False                      # full run
SEED = 0
MT_MODELS = {"600m": "facebook/nllb-200-distilled-600M", "3b": "facebook/nllb-200-3.3B"}

# Only the 3B strategies are listed: every 600m run already succeeded and is in
# results_v2.zip, so re-running them would waste ~4 h. To reproduce everything,
# restore the full lists kept in the comments below.
LLMS = {  # name: (repo, 4-bit?, strategies)
  "Qwen2.5-7B":   ("Qwen/Qwen2.5-7B-Instruct", False,
                   ["pivot_3b_seg", "pivot_3b_ctx"]),
                   # full: ["pivot_600m_seg","pivot_600m_ctx","pivot_600m_guard","pivot_3b_seg","pivot_3b_ctx"]
  "Llama-3.1-8B": ("unsloth/Meta-Llama-3.1-8B-Instruct", False,
                   ["pivot_3b_seg", "pivot_3b_ctx"]),
                   # full: ["pivot_600m_seg","pivot_600m_ctx","pivot_600m_guard","pivot_3b_seg","pivot_3b_ctx"]
  "Qwen2.5-14B":  ("Qwen/Qwen2.5-14B-Instruct", True,
                   ["pivot_3b_ctx"]),
                   # full: ["ar_direct","en_human","pivot_600m_seg","pivot_600m_ctx","pivot_3b_ctx"]
}
OUT = "/kaggle/working/results_v3"; os.makedirs(OUT, exist_ok=True)
ERRLOG = f"{OUT}/errors.txt"
HF_CACHE = os.path.expanduser("~/.cache/huggingface/hub")
def free_disk(repo):
    shutil.rmtree(os.path.join(HF_CACHE, "models--" + repo.replace("/", "--")), ignore_errors=True)
def safe(name, fn):
    try: fn(); return True
    except Exception as e:
        open(ERRLOG, "a", encoding="utf-8").write(f"=== {name}\n{traceback.format_exc()}\n")
        print("FAILED:", name, type(e).__name__, str(e)[:200]); gc.collect(); torch.cuda.empty_cache(); return False
print(torch.cuda.device_count(), "GPUs")

## 1. Data (same 900 Belebele and 1,500 ArabicMMLU questions as the first run)

In [ ]:
BB = "https://huggingface.co/datasets/facebook/belebele/resolve/main/data/{}.jsonl"
ar = pd.read_json(BB.format("arb_Arab"), lines=True); en = pd.read_json(BB.format("eng_Latn"), lines=True)
assert (ar.link == en.link).all() and (ar.question_number == en.question_number).all()
bb = pd.DataFrame({"id": range(900), "gold": ar.correct_answer_num.astype(int),
    "p_ar": ar.flores_passage, "q_ar": ar.question, "o_ar": ar[[f"mc_answer{i}" for i in range(1,5)]].astype(str).values.tolist(),
    "p_en": en.flores_passage, "q_en": en.question, "o_en": en[[f"mc_answer{i}" for i in range(1,5)]].astype(str).values.tolist()})
IDS = [1, 22, 34, 40, 46, 89, 93, 118, 119, 125, 135, 183, 186, 193, 197, 203, 215, 225, 228, 249, 254, 296, 299, 304, 306, 313, 328, 335, 348, 380, 404, 412, 420, 455, 459, 467, 469, 474, 475, 482, 483, 484, 487, 491, 501, 503, 516, 522, 536, 541, 544, 546, 548, 549, 556, 561, 565, 569, 570, 578, 585, 590, 618, 619, 637, 638, 651, 652, 656, 664, 665, 669, 675, 683, 695, 699, 711, 712, 740, 746, 766, 774, 777, 790, 798, 824, 834, 837, 846, 857, 866, 874, 890, 893, 901, 913, 918, 931, 944, 956, 962, 975, 995, 1021, 1027, 1032, 1033, 1040, 1041, 1044, 1047, 1053, 1060, 1067, 1080, 1082, 1085, 1095, 1096, 1099, 1128, 1129, 1131, 1143, 1172, 1185, 1189, 1204, 1205, 1209, 1211, 1223, 1239, 1244, 1251, 1252, 1258, 1260, 1263, 1276, 1291, 1294, 1295, 1310, 1313, 1317, 1325, 1331, 1346, 1347, 1353, 1354, 1356, 1370, 1374, 1388, 1400, 1411, 1421, 1435, 1448, 1453, 1461, 1472, 1483, 1493, 1495, 1511, 1525, 1539, 1548, 1549, 1552, 1557, 1559, 1564, 1569, 1570, 1576, 1579, 1592, 1620, 1622, 1623, 1624, 1628, 1631, 1632, 1639, 1645, 1648, 1652, 1653, 1679, 1686, 1688, 1694, 1701, 1706, 1718, 1732, 1743, 1754, 1763, 1771, 1784, 1785, 1792, 1806, 1810, 1819, 1828, 1833, 1842, 1844, 1857, 1866, 1879, 1887, 1890, 1896, 1950, 1962, 1975, 1976, 2006, 2015, 2028, 2054, 2055, 2083, 2093, 2112, 2122, 2149, 2154, 2155, 2157, 2164, 2165, 2170, 2175, 2181, 2197, 2220, 2239, 2271, 2283, 2322, 2333, 2342, 2345, 2351, 2398, 2400, 2403, 2422, 2426, 2436, 2464, 2480, 2533, 2542, 2553, 2558, 2583, 2584, 2585, 2596, 2609, 2613, 2615, 2617, 2618, 2633, 2657, 2670, 2680, 2700, 2708, 2710, 2724, 2728, 2733, 2756, 2761, 2762, 2763, 2770, 2771, 2784, 2814, 2827, 2839, 2841, 2846, 2847, 2858, 2859, 2869, 2901, 2904, 2991, 2994, 3010, 3022, 3032, 3035, 3040, 3046, 3054, 3079, 3080, 3095, 3096, 3102, 3125, 3132, 3141, 3158, 3168, 3235, 3247, 3271, 3277, 3282, 3303, 3309, 3334, 3355, 3359, 3385, 3418, 3420, 3437, 3440, 3446, 3459, 3470, 3476, 3488, 3499, 3502, 3542, 3543, 3558, 3573, 3578, 3583, 3601, 3620, 3627, 3629, 3638, 3646, 3647, 3651, 3666, 3669, 3675, 3688, 3693, 3697, 3702, 3707, 3721, 3728, 3729, 3750, 3752, 3754, 3756, 3760, 3766, 3772, 3784, 3790, 3803, 3809, 3814, 3829, 3842, 3864, 3865, 3881, 3897, 3920, 3930, 3942, 3956, 3959, 3966, 3971, 3976, 3982, 3986, 3990, 3992, 3998, 4003, 4009, 4029, 4032, 4039, 4043, 4051, 4061, 4079, 4082, 4102, 4132, 4135, 4141, 4147, 4154, 4163, 4165, 4187, 4208, 4210, 4218, 4234, 4237, 4250, 4259, 4268, 4285, 4288, 4289, 4297, 4299, 4333, 4336, 4343, 4365, 4368, 4369, 4376, 4388, 4415, 4432, 4435, 4447, 4458, 4469, 4473, 4477, 4489, 4507, 4511, 4520, 4523, 4527, 4529, 4544, 4548, 4556, 4562, 4577, 4584, 4588, 4589, 4591, 4596, 4600, 4601, 4608, 4611, 4616, 4621, 4626, 4630, 4641, 4677, 4678, 4692, 4694, 4695, 4713, 4715, 4757, 4765, 4782, 4812, 4825, 4831, 4858, 4875, 4884, 4887, 4891, 4896, 4897, 4921, 4931, 4932, 4948, 4956, 4961, 4994, 5006, 5008, 5009, 5014, 5020, 5026, 5031, 5036, 5039, 5048, 5079, 5104, 5113, 5115, 5122, 5124, 5135, 5136, 5139, 5143, 5144, 5148, 5149, 5153, 5163, 5166, 5174, 5175, 5208, 5214, 5216, 5219, 5225, 5237, 5246, 5249, 5255, 5261, 5265, 5308, 5334, 5335, 5351, 5387, 5399, 5409, 5413, 5419, 5431, 5442, 5445, 5466, 5471, 5508, 5513, 5514, 5516, 5524, 5526, 5529, 5551, 5570, 5596, 5598, 5620, 5628, 5636, 5681, 5683, 5690, 5698, 5707, 5719, 5732, 5738, 5751, 5752, 5759, 5761, 5764, 5780, 5783, 5784, 5785, 5790, 5793, 5799, 5800, 5805, 5816, 5834, 5839, 5846, 5852, 5853, 5856, 5879, 5886, 5897, 5901, 5905, 5906, 5911, 5932, 5935, 5940, 5945, 5978, 5984, 5995, 6008, 6014, 6073, 6087, 6100, 6104, 6126, 6131, 6159, 6165, 6168, 6170, 6173, 6188, 6191, 6202, 6203, 6210, 6211, 6212, 6224, 6227, 6234, 6249, 6250, 6254, 6259, 6262, 6263, 6264, 6277, 6280, 6298, 6299, 6300, 6312, 6322, 6326, 6328, 6331, 6335, 6338, 6353, 6359, 6368, 6370, 6379, 6387, 6394, 6399, 6403, 6406, 6408, 6419, 6423, 6426, 6431, 6436, 6441, 6442, 6452, 6454, 6468, 6470, 6474, 6478, 6480, 6484, 6491, 6500, 6504, 6505, 6510, 6511, 6519, 6521, 6522, 6529, 6550, 6551, 6555, 6560, 6562, 6567, 6578, 6594, 6599, 6605, 6630, 6642, 6645, 6660, 6666, 6677, 6679, 6684, 6686, 6688, 6690, 6692, 6694, 6695, 6704, 6711, 6713, 6717, 6720, 6726, 6728, 6732, 6734, 6740, 6742, 6744, 6759, 6764, 6765, 6773, 6776, 6780, 6783, 6784, 6800, 6802, 6803, 6808, 6818, 6819, 6822, 6824, 6829, 6832, 6834, 6835, 6836, 6847, 6851, 6863, 6869, 6870, 6873, 6892, 6895, 6896, 6900, 6902, 6908, 6909, 6912, 6919, 6935, 6942, 6948, 6954, 6956, 6975, 6990, 7015, 7021, 7029, 7043, 7047, 7048, 7062, 7072, 7073, 7088, 7099, 7120, 7182, 7183, 7193, 7220, 7249, 7261, 7265, 7268, 7294, 7300, 7312, 7333, 7339, 7349, 7356, 7377, 7383, 7393, 7401, 7416, 7417, 7429, 7447, 7455, 7464, 7472, 7494, 7514, 7521, 7535, 7539, 7555, 7557, 7560, 7564, 7565, 7574, 7581, 7590, 7592, 7594, 7607, 7608, 7614, 7621, 7642, 7653, 7670, 7697, 7725, 7737, 7744, 7753, 7759, 7768, 7778, 7785, 7795, 7796, 7816, 7821, 7852, 7855, 7857, 7862, 7870, 7892, 7893, 7906, 7914, 7931, 7936, 7950, 7958, 7967, 7969, 7981, 7996, 8004, 8010, 8028, 8038, 8051, 8071, 8088, 8115, 8177, 8184, 8197, 8200, 8212, 8221, 8227, 8247, 8293, 8323, 8337, 8338, 8349, 8352, 8357, 8390, 8413, 8418, 8459, 8465, 8479, 8503, 8515, 8540, 8567, 8586, 8587, 8604, 8628, 8630, 8653, 8654, 8665, 8675, 8682, 8688, 8699, 8701, 8738, 8745, 8751, 8765, 8775, 8794, 8795, 8803, 8828, 8832, 8837, 8867, 8873, 8882, 8888, 8893, 8895, 8930, 8935, 8951, 8973, 8992, 9052, 9054, 9063, 9064, 9097, 9101, 9109, 9118, 9121, 9123, 9144, 9148, 9154, 9172, 9174, 9187, 9190, 9194, 9203, 9207, 9216, 9219, 9239, 9241, 9260, 9262, 9282, 9295, 9306, 9310, 9312, 9315, 9324, 9354, 9366, 9370, 9381, 9387, 9401, 9404, 9405, 9407, 9431, 9441, 9442, 9457, 9463, 9465, 9480, 9483, 9511, 9545, 9547, 9557, 9591, 9592, 9615, 9627, 9636, 9655, 9656, 9659, 9660, 9673, 9684, 9688, 9698, 9726, 9729, 9733, 9750, 9766, 9786, 9787, 9801, 9806, 9808, 9816, 9825, 9828, 9842, 9852, 9863, 9877, 9884, 9907, 9920, 9945, 9949, 9953, 9978, 9983, 9988, 10001, 10021, 10039, 10042, 10050, 10067, 10078, 10119, 10125, 10145, 10157, 10162, 10163, 10196, 10202, 10205, 10207, 10223, 10228, 10243, 10246, 10253, 10258, 10265, 10272, 10291, 10307, 10309, 10321, 10342, 10365, 10380, 10400, 10401, 10419, 10424, 10436, 10437, 10439, 10448, 10456, 10460, 10464, 10470, 10488, 10489, 10493, 10495, 10503, 10512, 10515, 10516, 10522, 10531, 10565, 10568, 10595, 10600, 10614, 10625, 10644, 10672, 10681, 10690, 10698, 10700, 10701, 10725, 10738, 10753, 10758, 10761, 10768, 10769, 10773, 10782, 10807, 10811, 10814, 10818, 10823, 10824, 10825, 10832, 10846, 10847, 10859, 10918, 10940, 10945, 10961, 10996, 11013, 11023, 11029, 11053, 11073, 11110, 11113, 11115, 11117, 11131, 11132, 11144, 11148, 11166, 11181, 11226, 11251, 11258, 11262, 11271, 11275, 11279, 11287, 11292, 11304, 11340, 11341, 11347, 11357, 11361, 11363, 11372, 11394, 11402, 11408, 11416, 11420, 11428, 11450, 11485, 11502, 11525, 11526, 11537, 11555, 11563, 11588, 11589, 11598, 11601, 11602, 11611, 11651, 11659, 11670, 11700, 11702, 11706, 11718, 11725, 11730, 11737, 11738, 11740, 11742, 11758, 11776, 11785, 11790, 11806, 11810, 11815, 11833, 11835, 11838, 11841, 11857, 11860, 11869, 11878, 11879, 11880, 11891, 11894, 11897, 11900, 11907, 11912, 11915, 11916, 11921, 11930, 11937, 11939, 11957, 11974, 11978, 11980, 11982, 11986, 11997, 12006, 12008, 12014, 12016, 12038, 12041, 12046, 12047, 12048, 12051, 12055, 12061, 12064, 12069, 12078, 12080, 12086, 12087, 12089, 12090, 12093, 12095, 12104, 12108, 12110, 12117, 12120, 12122, 12139, 12140, 12143, 12146, 12149, 12154, 12162, 12175, 12181, 12183, 12185, 12205, 12207, 12212, 12214, 12230, 12234, 12235, 12241, 12244, 12247, 12252, 12254, 12262, 12265, 12270, 12273, 12276, 12278, 12287, 12291, 12294, 12297, 12299, 12304, 12315, 12320, 12327, 12339, 12341, 12348, 12351, 12354, 12362, 12366, 12374, 12375, 12381, 12394, 12399, 12404, 12420, 12430, 12448, 12449, 12452, 12465, 12474, 12476, 12482, 12484, 12485, 12490, 12500, 12506, 12518, 12520, 12527, 12528, 12533, 12544, 12549, 12563, 12565, 12568, 12574, 12575, 12576, 12579, 12584, 12585, 12587, 12588, 12590, 12593, 12595, 12596, 12598, 12602, 12603, 12613, 12622, 12630, 12632, 12637, 12649, 12652, 12653, 12665, 12682, 12684, 12704, 12742, 12767, 12770, 12777, 12813, 12822, 12829, 12843, 12844, 12854, 12862, 12877, 12888, 12905, 12911, 12941, 12965, 12980, 13030, 13033, 13037, 13040, 13048, 13055, 13058, 13070, 13121, 13140, 13145, 13154, 13159, 13167, 13180, 13181, 13201, 13203, 13215, 13231, 13254, 13260, 13267, 13278, 13289, 13291, 13326, 13334, 13341, 13347, 13397, 13415, 13424, 13446, 13459, 13475, 13486, 13493, 13500, 13504, 13510, 13520, 13524, 13525, 13528, 13529, 13556, 13563, 13575, 13577, 13589, 13606, 13610, 13620, 13628, 13630, 13658, 13675, 13679, 13685, 13694, 13695, 13696, 13715, 13716, 13734, 13776, 13790, 13807, 13813, 13827, 13851, 13852, 13856, 13868, 13873, 13879, 13889, 13899, 13904, 13924, 13926, 13928, 13944, 13987, 14004, 14006, 14043, 14045, 14049, 14077, 14078, 14080, 14095, 14098, 14101, 14103, 14109, 14139, 14147, 14167, 14176, 14177, 14200, 14202, 14220, 14237, 14267, 14281, 14287, 14298, 14301, 14304, 14307, 14311, 14321, 14325, 14339, 14342, 14355, 14368, 14373, 14379, 14380, 14403, 14410, 14411, 14433, 14446, 14471, 14472, 14476, 14477, 14499, 14505, 14509, 14519, 14528, 14534, 14546, 14562]
am = pd.read_csv("https://huggingface.co/datasets/MBZUAI/ArabicMMLU/resolve/main/All/test.csv")
am = am[am.ID.isin(IDS)].copy(); assert len(am) == 1500
am["gold"] = am["Answer Key"].map({"A":1,"B":2,"C":3,"D":4,"E":5}).astype(int)
am["o_ar"] = am[[f"Option {i}" for i in range(1,6)]].apply(lambda r: [str(x) for x in r if pd.notna(x)], axis=1)
am["p_ar"] = am["Context"].fillna("").astype(str); am["q_ar"] = am["Question"].astype(str)
am = am.rename(columns={"ID": "id", "Group": "group", "Subject": "subject"})[["id","gold","p_ar","q_ar","o_ar","group","subject"]]
if SMOKE:
    bb = bb.sample(30, random_state=SEED).reset_index(drop=True); am = am.groupby("group").head(6).reset_index(drop=True)
print(len(bb), len(am))

## 2. Translation: segment-level vs context-aware, 600M vs 3.3B

In [ ]:
SENT = re.compile(r'(?<=[\.\!\?؟])\s+')
AR = re.compile(r'[؀-ۿ]')
PRON = re.compile(r"\b(I|I'm|you|You|me|we|We|it's|okay|Okay|sorry)\b")
def opts_block(q, o): return q + "\n" + "\n".join(f"{i+1}. {x}" for i, x in enumerate(o))
def parse_block(out, n):
    parts = re.split(r'(?:^|\s)([1-9])[\.\)]\s*', out)
    nums, opts = parts[1::2], [p.strip().rstrip('.').strip() for p in parts[2::2]]
    if nums == [str(i+1) for i in range(n)] and all(opts): return parts[0].strip(), opts
    return None
def guard(src, mt):   # plain-code check: True = keep the Arabic original
    sw, mw = len(src.split()), len(mt.split())
    if not AR.search(src): return True
    return sw <= 2 and (mw > 2*sw + 1 or bool(PRON.search(mt)))

def translate(repo, texts, bs):
    tok = AutoTokenizer.from_pretrained(repo, src_lang="arb_Arab")
    # v2 OOMed here: 13.41 GiB allocated on a 14.56 GiB T4. That is either fp32 weights
    # (13.2 GB, if the deprecated `torch_dtype=` was ignored) or fp16 weights (6.6 GB)
    # plus beam-search activations at batch 16. `dtype=` is the current spelling and
    # the assert below makes the precision explicit either way.
    m = AutoModelForSeq2SeqLM.from_pretrained(repo, dtype=torch.float16,
                                              low_cpu_mem_usage=True).to("cuda:0").eval()
    gib = torch.cuda.memory_allocated(0) / 2**30
    print(f"  loaded {repo}: {gib:.2f} GiB on cuda:0", flush=True)
    assert next(m.parameters()).dtype == torch.float16, "model is not fp16 - it will OOM"
    eng = tok.convert_tokens_to_ids("eng_Latn"); uniq = sorted(set(t for t in texts if t))
    lens = {t: len(tok(t).input_ids) for t in uniq}; order = sorted(uniq, key=lambda t: lens[t]); out = {}
    torch.cuda.synchronize(); t0 = time.time()
    for k in range(0, len(order), bs):
        batch = order[k:k+bs]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=480).to("cuda:0")
        with torch.no_grad():
            g = m.generate(**enc, forced_bos_token_id=eng, num_beams=2, max_new_tokens=int(enc.input_ids.shape[1]*1.6)+16)
        out.update(zip(batch, tok.batch_decode(g, skip_special_tokens=True)))
        del enc, g
        if (k // bs) % 25 == 0:
            torch.cuda.empty_cache()
            print(f"    {min(k+bs, len(order))}/{len(order)}  {time.time()-t0:.0f}s", flush=True)
    torch.cuda.synchronize(); dt = time.time() - t0
    out[""] = ""; del m, tok; gc.collect(); torch.cuda.empty_cache(); free_disk(repo)
    print(f"  freed, {torch.cuda.memory_allocated(0)/2**30:.2f} GiB still held", flush=True)
    return out, dt, lens

def build(df, tag, repo, bs):
    sents = lambda s: [x for x in SENT.split(s) if x.strip()]
    seg_units = [x for s in df.p_ar for x in sents(s)] + list(df.q_ar) + [o for l in df.o_ar for o in l]
    blocks = [opts_block(q, o) for q, o in zip(df.q_ar, df.o_ar)]
    T, dt, lens = translate(repo, seg_units + blocks + list(df.p_ar), bs)
    seg_p = [" ".join(T[x] for x in sents(s)) for s in df.p_ar]
    df[f"p_{tag}_seg"] = seg_p; df[f"q_{tag}_seg"] = [T[q] for q in df.q_ar]
    df[f"o_{tag}_seg"] = [[T[o] for o in l] for l in df.o_ar]
    # context-aware: whole passage when it fits, question + options as one block (fallback to segments)
    df[f"p_{tag}_ctx"] = [T[s] if s and lens.get(s, 999) <= 450 else sp for s, sp in zip(df.p_ar, seg_p)]
    qc, oc, ok = [], [], []
    for b, q, l in zip(blocks, df.q_ar, df.o_ar):
        r = parse_block(T[b], len(l))
        if r: qc.append(r[0] or T[q]); oc.append(r[1]); ok.append(1)
        else: qc.append(T[q]); oc.append([T[o] for o in l]); ok.append(0)
    df[f"q_{tag}_ctx"], df[f"o_{tag}_ctx"], df[f"block_ok_{tag}"] = qc, oc, ok
    if tag == "600m":
        df["o_600m_guard"] = [[a if guard(a, m) else m for a, m in zip(la, lm)] for la, lm in zip(df.o_ar, df.o_600m_seg)]
        df["guard_flags"] = [sum(guard(a, m) for a, m in zip(la, lm)) for la, lm in zip(df.o_ar, df.o_600m_seg)]
        df["p_600m_guard"], df["q_600m_guard"] = df.p_600m_seg, df.q_600m_seg
    return dt

timing = {}
for tag, repo in MT_MODELS.items():
    both = pd.concat([bb.assign(_ds="bb"), am.assign(_ds="am")], ignore_index=True)
    def job():
        # 3B at batch 8: fp16 weights are 6.6 GB, leaving room for beam search on a T4
        timing[tag] = build(both, tag, repo, 48 if tag == "600m" else 8)
        for c in both.columns:
            if c not in bb.columns and c != "_ds":
                bb[c] = both[both._ds == "bb"][c].values; am[c] = both[both._ds == "am"][c].values
    safe(f"MT {tag}", job)
json.dump(timing, open(f"{OUT}/mt_timing_v3.json", "w"))
bb.to_parquet(f"{OUT}/belebele_translations_v3.parquet"); am.to_parquet(f"{OUT}/ammlu_translations_v3.parquet")
print(timing, "| 3b block parsed (bb):", bb.get("block_ok_3b", pd.Series([0])).mean())

## 3. Prompts, parsing, OOM-safe runner

In [ ]:
def opts(o): return "\n".join(f"{i+1}. {x}" for i, x in enumerate(o))
def p_en(p, q, o):
    head = f"Passage: {p}\n\n" if p else ""
    return f"Answer the question by choosing the number of the correct option.\n\n{head}Question: {q}\n{opts(o)}\n\nReply with the number of the correct option only."
def p_ar(p, q, o):
    head = f"النص: {p}\n\n" if p else ""
    return f"أجب عن السؤال باختيار رقم الخيار الصحيح.\n\n{head}السؤال: {q}\n{opts(o)}\n\nأجب برقم الخيار الصحيح فقط."
AR_DIG = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
def parse(t, k):
    d = [int(x) for x in re.findall(r'([1-9])', (t or "").translate(AR_DIG)) if 1 <= int(x) <= k]
    return d[0] if d else -1
def prompts_for(df, strat):
    if strat == "ar_direct": return [p_ar(r.p_ar, r.q_ar, r.o_ar) for r in df.itertuples()]
    if strat == "en_human":  return [p_en(r.p_en, r.q_en, r.o_en) for r in df.itertuples()]
    s = strat.replace("pivot_", "")
    return [p_en(getattr(r, f"p_{s}"), getattr(r, f"q_{s}"), getattr(r, f"o_{s}")) for r in df.itertuples()]

def load(repo, four_bit):
    tok = AutoTokenizer.from_pretrained(repo, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(device_map="auto")
    # torch_dtype is kept EXACTLY as the completed 600m runs used it. Changing it to the
    # current `dtype=` spelling could silently switch precision and make the new 3B runs
    # non-comparable with results_v2. The print below just records what actually loaded.
    if four_bit: kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    else: kw["torch_dtype"] = torch.float16
    m = AutoModelForCausalLM.from_pretrained(repo, **kw).eval()
    print(f"  {repo}: param dtype={next(m.parameters()).dtype}, "
          f"{torch.cuda.memory_allocated(0)/2**30:.2f} GiB on cuda:0", flush=True)
    for a in ("temperature", "top_p", "top_k"): setattr(m.generation_config, a, None)
    return tok, m

def gen_batch(tok, m, texts, max_new):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad(): out = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
    torch.cuda.synchronize(); dt = time.time() - t0
    gen = out[:, enc.input_ids.shape[1]:]; res = []
    for j in range(len(texts)):
        g = gen[j]; stop = (g == tok.pad_token_id) | (g == tok.eos_token_id)
        n = int(stop.nonzero()[0, 0]) if stop.any() else int(g.shape[0])
        res.append(dict(prompt_tokens=int(enc.attention_mask[j].sum()), gen_tokens=n,
                        output=tok.decode(g[:n], skip_special_tokens=True), batch_seconds=dt, batch_size=len(texts)))
    return res

def run(tok, m, prompts, max_new=16, budget=6000):
    texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in prompts]
    lens = [len(tok(t, add_special_tokens=False).input_ids) for t in texts]
    order = sorted(range(len(texts)), key=lambda i: -lens[i]); res = [None]*len(texts); k = 0
    while k < len(order):
        bs = max(1, min(16, budget // max(lens[order[k]], 1)))
        while True:
            idx = order[k:k+bs]
            try:
                for i, r in zip(idx, gen_batch(tok, m, [texts[i] for i in idx], max_new)): res[i] = r
                break
            except torch.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if bs == 1: raise
                bs = max(1, bs // 2)
        k += len(idx)
    return res

## 4. Runs (number-only answers)

In [ ]:
for name, (repo, four_bit, strategies) in LLMS.items():
    t0 = time.time()
    def model_job():
        tok, m = load(repo, four_bit)
        for ds_name, df in [("belebele", bb), ("ammlu", am)]:
            for strat in strategies:
                if strat == "en_human" and ds_name == "ammlu": continue
                fn = f"{OUT}/{name}__{ds_name}__{strat}.parquet"
                if os.path.exists(fn): continue
                def job():
                    pr = prompts_for(df, strat); res = run(tok, m, pr)
                    k = [len(o) for o in df.o_ar]
                    rows = [dict(model=name, dataset=ds_name, strategy=strat, id=r.id, gold=int(r.gold),
                                 pred=parse(x["output"], kk), prompt=p, **x) for r, p, x, kk in zip(df.itertuples(), pr, res, k)]
                    out = pd.DataFrame(rows)
                    if ds_name == "ammlu": out["group"] = df.group.values; out["subject"] = df.subject.values
                    out.to_parquet(fn); print(name, ds_name, strat, "acc", round((out.pred == out.gold).mean(), 3))
                safe(f"{name} {ds_name} {strat}", job)
        del m; gc.collect(); torch.cuda.empty_cache()
    safe(f"MODEL {name}", model_job); free_disk(repo)
    print(f"== {name} done in {(time.time()-t0)/60:.1f} min")

In [ ]:
with zipfile.ZipFile("/kaggle/working/results_v3.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT): z.write(f"{OUT}/{f}", f)
print(sorted(os.listdir(OUT)))